# Radiant RAG MCP — Audio Ingestion & Summarization Test Notebook

End-to-end test of the `ingest_audio` MCP tool and `VideoSummarizationAgent`
applied to audio-only files.  Synthetic test audio files are generated in-process
using `espeak` TTS so the suite runs without any external media.

| | |
|---|---|
| **Transport** | Streamable HTTP |
| **Storage** | ChromaDB (embedded) |
| **Transcription** | Whisper (`base` model) |
| **Summarization** | `VideoSummarizationAgent` (brief / standard / detailed) |
| **LLM** | Ollama Cloud — `gemma4:31b-cloud` |

**Sections that require an LLM key are marked `[LLM]`.**  
Add `OLLAMA_API_KEY` to Colab Secrets (key icon in the left sidebar) to enable them.

---

## Run order

```
1a → (auto restart) → 1b → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11
→ [LLM] 12 → [LLM] 13 → [LLM] 14 → [LLM] 15 → [LLM] 16 → 17 → Summary
```


## 1  Install

Installs `radiant-rag-mcp` with the ChromaDB backend, audio dependencies
(faster-whisper, ffmpeg-python), embedding/reranking models, and helpers.

The cell ends with a **kernel restart** so freshly installed `.so` files are
loaded correctly.  After the restart, continue from **cell 1b**.


In [ ]:
import subprocess, sys, os

CONFIG_PATH = '/content/config.yaml'

# ── Step 1: FFmpeg (standard apt version is sufficient for audio) ──────────
!apt-get install -y -q ffmpeg espeak
!ffmpeg -version 2>&1 | head -1

# ── Step 2: Detect PyTorch CUDA tag ──────────────────────────────────────
import torch as _torch
_torch_cuda = _torch.version.cuda or ''
_cu_tag = f"cu{_torch_cuda.replace('.','')[:3]}" if _torch_cuda else ''
print(f'torch {_torch.__version__}  CUDA {_torch_cuda}  ->  tag: {_cu_tag}')
del _torch

# ── Step 3: Core RAG package ──────────────────────────────────────────────
!pip install -q --upgrade --no-cache-dir "radiant-rag-mcp[chroma] @ git+https://github.com/dshipley71/radiant-rag-mcp.git"
!pip install -q --prefer-binary nest_asyncio httpx "fastmcp>=3.0"
!wget -q "https://raw.githubusercontent.com/dshipley71/radiant-rag-mcp/main/config.yaml" -O {CONFIG_PATH}

# ── Step 4: Audio extras ──────────────────────────────────────────────────
!pip install -q ffmpeg-python "faster-whisper>=1.0.0"

# ── Step 5: Repair numpy/scipy/sklearn ───────────────────────────────────
!pip install -q --force-reinstall numpy scipy scikit-learn
!pip install -q --upgrade "transformers>=4.45.0" "sentence-transformers>=2.7"

# ── Step 6: Pillow ────────────────────────────────────────────────────────
!pip install -q --force-reinstall "Pillow>=11.0"

# ── Step 7: Restart ──────────────────────────────────────────────────────
print('All installs complete. Restarting kernel ...')
import time; time.sleep(1)
os.kill(os.getpid(), 9)


Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  espeak-data libespeak1 libportaudio2 libsonic0
The following NEW packages will be installed:
  espeak espeak-data libespeak1 libportaudio2 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 42 not upgraded.
Need to get 1,382 kB of archives.
After this operation, 3,178 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudio2 amd64 19.6.0-1.1 [65.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 espeak-data amd64 1.48.15+dfsg-3 [1,085 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libespeak1 amd64 1.48.15+dfsg-3 [156 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 espeak amd64 1.48.

### ↻ Kernel restarted — continue from cell 1b

All package installs completed. Run **cell 1b** to verify imports and pre-cache models.


In [1]:
# ── Cell 1b: runs after the automatic kernel restart ─────────────────────
import sys, warnings
from pathlib import Path

CONFIG_PATH = '/content/config.yaml'

# 1 ── Verify numpy ────────────────────────────────────────────────────────
import numpy as np
print(f'numpy  {np.__version__}')
try:
    import numpy._core.strings
    print('numpy  OK')
except (ImportError, AttributeError) as _e:
    raise RuntimeError(f'numpy inconsistent ({_e}). Re-run 1a → restart → re-run 1b.')

# 2 ── Verify scipy ────────────────────────────────────────────────────────
import scipy
print(f'scipy  {scipy.__version__}  OK')

# 3 ── Verify Pillow ───────────────────────────────────────────────────────
import PIL
print(f'Pillow  {PIL.__version__}  OK')

# 4 ── Pre-cache embedding / reranking models ─────────────────────────────
from sentence_transformers import SentenceTransformer, CrossEncoder
SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
print('sentence-transformers  OK')
CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2')
print('cross-encoder  OK')

print('\nInstall complete')


numpy  2.4.4
numpy  OK
scipy  1.17.1  OK
Pillow  12.2.0  OK


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

cross-encoder  OK

Install complete


## 2  Import checks

Verifies every dependency the audio pipeline needs is importable.
A `MISSING` line means the package failed to install in section 1.


In [2]:
import sys
import importlib
from pathlib import Path

# ── Required third-party packages ─────────────────────────────────────────
REQUIRED = [
    ('fastmcp',               'fastmcp'),
    ('nest_asyncio',          'nest_asyncio'),
    ('chromadb',              'chromadb'),
    ('sentence_transformers', 'sentence-transformers'),
    ('faster_whisper',        'faster-whisper'),
    ('ffmpeg',                'ffmpeg-python'),
]

# ── Radiant audio submodules ──────────────────────────────────────────────
AUDIO_CORE = [
    ('radiant_rag_mcp',                            'radiant-rag-mcp'),
    ('radiant_rag_mcp.ingestion.video_processor',  'see diagnostic below'),
    ('radiant_rag_mcp.agents.video_summarization', 'see diagnostic below'),
]

_missing_required = []
_missing_audio    = {}

print('=== Required packages ===')
for module, pkg in REQUIRED:
    try:
        __import__(module)
        print(f'  ✓       {module}')
    except ImportError:
        print(f'  ✗  {module}  ->  pip install {pkg}')
        _missing_required.append(pkg)

print()
print('=== Radiant RAG audio submodules ===')
for module, _ in AUDIO_CORE:
    try:
        __import__(module)
        print(f'  ✓       {module}')
    except Exception as e:
        print(f'  ✗     {module}')
        _missing_audio[module] = e

if _missing_audio:
    print()
    for mod, exc in _missing_audio.items():
        print(f'  Module : {mod}')
        print(f'  Error  : {type(exc).__name__}: {exc}')
        cause = exc.__cause__ or exc.__context__
        if cause:
            print(f'  Caused by: {type(cause).__name__}: {cause}')
        print()

if _missing_required:
    raise ImportError(f'Missing required packages: {", ".join(_missing_required)}')

if _missing_audio:
    import warnings
    warnings.warn(f'Audio submodules not importable: {list(_missing_audio)}. Sections 8-16 will fail.',
                  stacklevel=1)
else:
    print()
    print('All imports ok')


=== Required packages ===
  ✓       fastmcp
  ✓       nest_asyncio
  ✓       chromadb
  ✓       sentence_transformers
  ✓       faster_whisper
  ✓       ffmpeg

=== Radiant RAG audio submodules ===
  ✓       radiant_rag_mcp


  ✓       radiant_rag_mcp.ingestion.video_processor
  ✓       radiant_rag_mcp.agents.video_summarization

All imports ok


## 3  Configuration

Sets every environment variable the server needs before startup.
Audio-specific keys use the prefix `RADIANT_VIDEO_WHISPER_*`.

> Add `OLLAMA_API_KEY` to **Colab Secrets** (key icon, left sidebar) before running.


In [3]:
import os

# Read LLM key from Colab Secrets, falling back to env
try:
    from google.colab import userdata
    _key = userdata.get('OLLAMA_API_KEY') or ''
except Exception:
    _key = os.environ.get('RADIANT_OLLAMA_API_KEY', '')

os.environ.update({
    'RADIANT_OLLAMA_BASE_URL':               'https://ollama.com/v1',
    'RADIANT_OLLAMA_API_KEY':                _key,
    'RADIANT_TRANSPORT':                     'http',
    'RADIANT_HOST':                          '127.0.0.1',
    'RADIANT_PORT':                          '8765',
    'RADIANT_STORAGE_BACKEND':               'chroma',
    'RADIANT_CONFIG_PATH':                   CONFIG_PATH,
    # Pipeline
    'RADIANT_PIPELINE_USE_CRITIC':           'false',
    'RADIANT_CITATION_ENABLED':              'false',
    'RADIANT_LLM_BACKEND_TIMEOUT':           '120',
    'RADIANT_LLM_BACKEND_MAX_RETRIES':       '0',
    'RADIANT_RERANKING_BACKEND_DEVICE':      'cuda',
    'RADIANT_EMBEDDING_BACKEND_DEVICE':      'cuda',
    'RADIANT_CHUNKING_USE_LLM_CHUNKING':     'false',
    'RADIANT_RETRIEVAL_DENSE_TOP_K':         '5',
    'RADIANT_RETRIEVAL_BM25_TOP_K':          '5',
    # Whisper transcription settings
    'RADIANT_VIDEO_WHISPER_MODEL':           'base',
    'RADIANT_VIDEO_WHISPER_DEVICE':          'auto',
    'RADIANT_VIDEO_WHISPER_COMPUTE_TYPE':    'int8',
    'RADIANT_VIDEO_WHISPER_LANGUAGE':        'auto',
    # Summarization defaults (overridden per-cell in sections 13-16)
    'RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL':           'standard',
    'RADIANT_VIDEO_SUMMARIZATION_WINDOW_CAPTION_SENTENCES': '4',
})

SERVER_URL  = f"http://{os.environ['RADIANT_HOST']}:{os.environ['RADIANT_PORT']}/mcp"
HAS_LLM_KEY = bool(_key)

print('=== Configuration ===')
print(f'  Server URL      : {SERVER_URL}')
print(f'  LLM key set     : {HAS_LLM_KEY}')
print(f'  Whisper model   : {os.environ["RADIANT_VIDEO_WHISPER_MODEL"]}')
print(f'  Whisper device  : {os.environ["RADIANT_VIDEO_WHISPER_DEVICE"]}')
print(f'  Summary detail  : {os.environ["RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL"]}')

if not HAS_LLM_KEY:
    print()
    print('[NOTE] OLLAMA_API_KEY not found. [LLM] cells will be skipped.')
    print('       Add it to Colab Secrets (key icon, left sidebar).')


=== Configuration ===
  Server URL      : http://127.0.0.1:8765/mcp
  LLM key set     : True
  Whisper model   : base
  Whisper device  : auto
  Summary detail  : standard


## 4  Shared helpers

Defines:
- `run(tool, args)` — call an MCP tool and pretty-print the JSON result
- `assert_ok(result)` — raise `AssertionError` on tool errors
- `skip_llm()` — guard used in every `[LLM]` cell
- `_wait_for_server()` — async HTTP probe used by section 5


In [4]:
import asyncio
import json
import logging
import threading
import time
import textwrap
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()  # Allow asyncio.run() inside Jupyter/Colab event loop

import httpx as _httpx
from fastmcp import Client
from fastmcp.exceptions import ToolError


async def _call(tool: str, args: dict = None):
    """Async helper: open a short-lived MCP client, call the tool, return parsed result."""
    try:
        async with Client(SERVER_URL) as client:
            raw = await client.call_tool(tool, args or {})
    except ToolError as e:
        return {'status': 'error', 'tool': tool, 'message': str(e)}
    except Exception as e:
        return {'status': 'error', 'tool': tool, 'message': f'{type(e).__name__}: {e}'}

    content = raw.content if hasattr(raw, 'content') else (raw or [])
    if not content:
        return None
    item = content[0]
    text = item.text if hasattr(item, 'text') else str(item)
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return text


def run(tool: str, args: dict = None):
    """Synchronous wrapper around _call; pretty-prints and returns the result."""
    result = asyncio.run(_call(tool, args))
    print(json.dumps(result, indent=2, default=str))
    return result


def skip_llm() -> bool:
    """Return True (and print a notice) when no LLM key is available."""
    if not HAS_LLM_KEY:
        print('[SKIPPED] Set OLLAMA_API_KEY in Colab Secrets to run this cell.')
        return True
    return False


def assert_ok(result, field: str = None):
    """Assert the tool result is not an error dict; optionally check a field exists."""
    assert result is not None, 'Tool returned no result'
    if isinstance(result, dict):
        if result.get('status') == 'error':
            raise AssertionError(f'Tool error: {result.get("message", result)}')
        if field:
            assert field in result, f'Expected field "{field}" missing from result: {result}'
    print('[OK]')


async def _wait_for_server(url: str, timeout: int = 120, interval: float = 1.0) -> bool:
    """Poll the MCP endpoint until it responds or the timeout expires."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            async with _httpx.AsyncClient() as http:
                await http.get(url, timeout=2.0,
                               headers={'Accept': 'application/json, text/event-stream'})
                return True
        except (_httpx.ConnectError, _httpx.ConnectTimeout, _httpx.ReadTimeout):
            await asyncio.sleep(interval)
    return False


print('Helpers loaded')


Helpers loaded


## 5  Server startup

Starts the Radiant RAG MCP server in a background daemon thread.
Tries `http` then `streamable-http` transport automatically.
Polls the endpoint for up to 90 s before raising `TimeoutError`.


In [5]:
import subprocess, time as _time

# Kill any process already bound to the port so restarts work cleanly
_port = int(os.environ['RADIANT_PORT'])
subprocess.run(['fuser', '-k', f'{_port}/tcp'], capture_output=True)
_time.sleep(1.0)

logging.basicConfig(stream=__import__('sys').stderr, level=logging.WARNING, force=True)

_server_ready   = threading.Event()
_server_error   = None
_transport_used = None


def _run_server():
    global _server_error, _transport_used
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        from radiant_rag_mcp.server import mcp
        print('Package  : radiant_rag_mcp  ok')
        _server_ready.set()
        host = os.environ['RADIANT_HOST']
        port = int(os.environ['RADIANT_PORT'])
        for _t in ['http', 'streamable-http']:
            try:
                _transport_used = _t
                mcp.run(transport=_t, host=host, port=port)
                return
            except Exception as _e:
                if 'Unknown transport' in str(_e) or 'unknown transport' in str(_e).lower():
                    continue
                raise
        raise RuntimeError('No supported HTTP transport found (tried: http, streamable-http)')
    except Exception as exc:
        _server_error = exc
        if not _server_ready.is_set():
            _server_ready.set()


_thread = threading.Thread(target=_run_server, name='radiant-mcp', daemon=True)
_thread.start()

if not _server_ready.wait(timeout=30):
    raise TimeoutError('Server thread did not signal ready within 30 s.')
if _server_error:
    raise _server_error

print(f'Waiting for server at {SERVER_URL} ...')
_deadline = time.time() + 90
while time.time() < _deadline:
    if _server_error:
        raise RuntimeError(f'Server thread failed: {_server_error}')
    if asyncio.run(_wait_for_server(SERVER_URL, timeout=5)):
        print(f'Server ready  ->  {SERVER_URL}  (transport: {_transport_used})')
        break
    time.sleep(1)
else:
    raise TimeoutError('Server did not bind within 90 s.')


Package  : radiant_rag_mcp  ok
Waiting for server at http://127.0.0.1:8765/mcp ...


╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.2.4                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      radiant-rag, 3.2.4                          │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

2026-04-24 15:01:08,847 - radiant_rag_mcp.app - INFO - Initializing Radiant RAG...
2026-04-24 15:01:08,847 - radiant_rag_mcp.llm.client - INFO - Using new backend configuration system
2026-04-24 15:01:08,852 - radiant_rag_mcp.llm.backends.factory - INFO - Creating LLM backend: type=ollama
2026-04-24 15:01:08,853 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Configured OpenAI-compatible LLM backend: model=gemma3:12b-cloud, base_url=https://ollama.com/v1
2026-04-24 15:01:08,854 - radiant_rag_mcp.llm.backends.factory - INFO - Creating embedding backend: type=local
2026-04-24 15:01:08,855 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Initializing sentence-transformers embedding backend: sentence-transformers/all-MiniLM-L12-v2
[transformers] The following layers were not sharded: encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.self.query.weight, embeddings.LayerNorm.bias, pooler.dense.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.l

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/llm/backends/embedding_backends.py:104: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._embedding_dim = self._model.get_sentence_embedding_dimension()
2026-04-24 15:01:11,162 - radiant_rag_mcp.utils.cache - INFO - Initialized EmbeddingCache with max_size=10000 (true LRU)
2026-04-24 15:01:11,163 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Loaded sentence-transformers embedding model: sentence-transformers/all-MiniLM-L12-v2 (dim=384, device=cuda)
2026-04-24 15:01:11,164 - radiant_rag_mcp.llm.backends.factory - INFO - Creating reranking backend: type=local
2026-04-24 15:01:11,165 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Initializing cross-encoder reranking backend: cross-encoder/ms-marco-MiniLM-L12-v2
[transformers] The following layers were not sharded: bert.encoder.layer.*.attention.output.dense.weight, bert.encoder.layer.*.attentio

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-04-24 15:01:12,852 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Loaded cross-encoder reranking model: cross-encoder/ms-marco-MiniLM-L12-v2 (device=cuda)
2026-04-24 15:01:12,856 - radiant_rag_mcp.storage.factory - INFO - Creating Chroma vector store
2026-04-24 15:01:13,012 - radiant_rag_mcp.storage.chroma_store - INFO - Initialized ChromaDB store at 'data/chroma_db' with collection 'radiant_docs'
2026-04-24 15:01:13,014 - radiant_rag_mcp.utils.conversation - INFO - ConversationStore: storage backend is 'chroma' — using in-memory storage
2026-04-24 15:01:13,029 - radiant_rag_mcp.utils.metrics_export - INFO - Prometheus metrics exporter initialized
2026-04-24 15:01:13,030 - PlanningAgent - INFO - [999fedf2] [enabled=True] Initialized PlanningAgent
2026-04-24 15:01:13,030 - QueryDecompositionAgent - INFO - [167abb19] [enabled=True] Initialized QueryDecompositionAgent
2026-04-24 15:01:13,032 - QueryRewriteAgent - INFO - [362141f1] [enabled=True] Initialized QueryRewriteAg

[04/24/26 15:01:13] INFO     Starting MCP server 'radiant-rag' with transport 'http' on            ]8;id=917457;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=509454;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#301\301]8;;\
                             http://127.0.0.1:8765/mcp                                                             

INFO:     Started server process [3253]
INFO:     Waiting for application startup.
2026-04-24 15:01:13,105 - mcp.server.streamable_http_manager - INFO - StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)
2026-04-24 15:01:15,072 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: eb726200a5ec41a19200541caae24c1d


INFO:     127.0.0.1:51678 - "GET /mcp HTTP/1.1" 400 Bad Request
Server ready  ->  http://127.0.0.1:8765/mcp  (transport: http)


## 6  Tool list verification

Lists every registered MCP tool and asserts that `ingest_audio` is present.
A missing tool means the audio sub-package did not install or register correctly.


In [6]:
async def _list_tools():
    async with Client(SERVER_URL) as client:
        return await client.list_tools()

tools      = asyncio.run(_list_tools())
registered = {t.name for t in tools}

print(f'{len(tools)} tool(s) registered:')
for t in sorted(tools, key=lambda x: x.name):
    desc = (t.description or '').splitlines()[0]
    print(f'  {t.name:<30}  {desc}')

print()
if 'ingest_audio' in registered:
    print('[ASSERT OK] ingest_audio is registered')
else:
    print('[FAIL] ingest_audio is NOT in the tool list.')
    print()
    print('  ► Re-run section 1, restart the runtime, and re-run sections 1-6.')
    raise AssertionError('ingest_audio not registered. See diagnostic above.')


2026-04-24 15:01:19,591 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: a87623222db947f3b17251325ac4ba5e


INFO:     127.0.0.1:51038 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:01:20,031 - mcp.client.streamable_http - INFO - Received session ID: a87623222db947f3b17251325ac4ba5e
2026-04-24 15:01:20,034 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:51052 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:51058 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:51072 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:01:20,056 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:01:20,068 - mcp.server.streamable_http - INFO - Terminating session: a87623222db947f3b17251325ac4ba5e


INFO:     127.0.0.1:51076 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-24 15:01:20,073 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


12 tool(s) registered:
  clear_index                     Delete all indexed documents from the knowledge base.
  get_index_stats                 Return document counts and system health for the knowledge base.
  ingest_audio                    Ingest one or more local audio-only files into the RAG knowledge base
  ingest_documents                Index local files or directories into the knowledge base.
  ingest_url                      Index a URL or GitHub repository into the knowledge base.
  ingest_video                    Ingest one or more video files or remote video URLs into the RAG knowledge base.
  query_knowledge                 Answer a question using the full agentic RAG pipeline.
  rebuild_bm25                    Rebuild the BM25 sparse index from the current vector store contents.
  search_knowledge                Retrieve documents from the knowledge base without LLM generation.
  set_ingest_mode                 Set the default ingestion storage mode for this server sess

## 7  Create synthetic test audio files

Generates three audio files entirely in-process using `espeak` TTS and `ffmpeg`:

| File | Format | Content |
|------|--------|--------|
| `lecture_en.wav` | WAV 16 kHz mono | Machine-learning lecture narration (~30 s) |
| `lecture_en.mp3` | MP3 128 kbps | Same content re-encoded via ffmpeg |
| `lecture_en.m4a` | AAC / M4A | Same content re-encoded via ffmpeg |

All three files share the same narration so that search results in section 11
appear across multiple sources.


In [7]:
import subprocess
from pathlib import Path

AUDIO_DIR   = Path('/tmp/radiant_audio_test')
AUDIO_DIR.mkdir(exist_ok=True)

WAV_FILE  = str(AUDIO_DIR / 'lecture_en.wav')
MP3_FILE  = str(AUDIO_DIR / 'lecture_en.mp3')
M4A_FILE  = str(AUDIO_DIR / 'lecture_en.m4a')

# Narration text — long enough to produce multiple Whisper segments and chunks
_NARRATION = (
    'Introduction to machine learning. '
    'Machine learning is a branch of artificial intelligence that enables computers '
    'to learn from data without being explicitly programmed. '
    'There are three main paradigms: supervised learning, unsupervised learning, '
    'and reinforcement learning. '
    'In supervised learning, models are trained on labelled examples. '
    'The model learns to map inputs to outputs by minimising a loss function. '
    'Common supervised tasks include classification and regression. '
    'Unsupervised learning discovers hidden patterns in unlabelled data. '
    'Clustering algorithms group similar data points together. '
    'Dimensionality reduction techniques such as principal component analysis '
    'compress data into fewer dimensions while preserving structure. '
    'Reinforcement learning trains an agent to maximise cumulative reward '
    'through interaction with an environment. '
    'Neural networks are the core architecture behind modern machine learning. '
    'A neural network consists of layers of interconnected neurons. '
    'Each neuron applies a weighted sum followed by a non-linear activation function. '
    'Deep learning refers to neural networks with many hidden layers. '
    'Convolutional networks excel at image recognition tasks. '
    'Recurrent networks process sequential data such as time series and text. '
    'Transformers use self-attention mechanisms and power large language models. '
    'Training a neural network requires computing gradients via backpropagation. '
    'Optimisers such as stochastic gradient descent update the weights iteratively. '
    'Regularisation techniques like dropout prevent overfitting to the training set. '
    'Evaluation metrics differ by task: accuracy for classification, '
    'mean squared error for regression, and BLEU score for translation. '
    'This concludes the introduction to machine learning fundamentals.'
)

# ── Generate WAV via espeak TTS ──────────────────────────────────────────
print('Generating WAV with espeak TTS ...')
r = subprocess.run(
    ['espeak', '-s', '140', '-v', 'en', _NARRATION, '-w', WAV_FILE],
    capture_output=True
)
if r.returncode != 0 or not Path(WAV_FILE).exists():
    # fallback: voiced-frequency tones Whisper can detect
    print('  espeak failed — generating voiced-frequency WAV via ffmpeg ...')
    subprocess.run(
        ['ffmpeg', '-y', '-f', 'lavfi',
         '-i', 'aevalsrc=sin(2*PI*200*t)+0.5*sin(2*PI*400*t)+0.3*sin(2*PI*800*t):s=16000:d=30',
         WAV_FILE],
        capture_output=True, check=True
    )
    print('  fallback WAV  OK')
else:
    print('  espeak TTS  OK')

# ── Convert to MP3 ────────────────────────────────────────────────────────
print('Converting to MP3 ...')
subprocess.run(
    ['ffmpeg', '-y', '-i', WAV_FILE, '-codec:a', 'libmp3lame', '-b:a', '128k', MP3_FILE],
    capture_output=True, check=True
)
print('  MP3  OK')

# ── Convert to M4A / AAC ─────────────────────────────────────────────────
print('Converting to M4A ...')
subprocess.run(
    ['ffmpeg', '-y', '-i', WAV_FILE, '-codec:a', 'aac', '-b:a', '128k', M4A_FILE],
    capture_output=True, check=True
)
print('  M4A  OK')

print()
print('Test audio files created:')
for fp in [WAV_FILE, MP3_FILE, M4A_FILE]:
    p = Path(fp)
    print(f'  {p.name:<25}  {p.stat().st_size // 1024:>5} KB  ->  {fp}')

assert Path(WAV_FILE).exists(), f'WAV not created: {WAV_FILE}'
assert Path(MP3_FILE).exists(), f'MP3 not created: {MP3_FILE}'
assert Path(M4A_FILE).exists(), f'M4A not created: {M4A_FILE}'
print('[ASSERT OK] All three test audio files exist on disk')


Generating WAV with espeak TTS ...
  espeak TTS  OK
Converting to MP3 ...
  MP3  OK
Converting to M4A ...
  M4A  OK

Test audio files created:
  lecture_en.wav              5429 KB  ->  /tmp/radiant_audio_test/lecture_en.wav
  lecture_en.mp3              1971 KB  ->  /tmp/radiant_audio_test/lecture_en.mp3
  lecture_en.m4a              1594 KB  ->  /tmp/radiant_audio_test/lecture_en.m4a
[ASSERT OK] All three test audio files exist on disk


## 8  `_extract_audio_metadata()` smoke test

Calls `VideoProcessor._extract_audio_metadata()` and `VideoProcessor.is_audio_file()`
directly (no MCP round-trip) and asserts:
- All three files return `is_audio_file() == True`
- Metadata has `is_silent=False` and `duration > 0`
- An MP4 path returns `is_audio_file() == False`


In [8]:
from radiant_rag_mcp.ingestion.video_processor import VideoProcessor, AUDIO_EXTENSIONS
from radiant_rag_mcp.config import VideoProcessorConfig

cfg = VideoProcessorConfig()
vp  = VideoProcessor(config=cfg)

print('=== is_audio_file() ===')
audio_cases = [
    (WAV_FILE, True),
    (MP3_FILE, True),
    (M4A_FILE, True),
    ('/tmp/some_video.mp4', False),
]
for path, expected in audio_cases:
    result = vp.is_audio_file(path)
    status = 'ok' if result == expected else 'MISMATCH'
    print(f'  {status}  {Path(path).name:<25}  is_audio_file={result}  (expected {expected})')
    assert result == expected, f'is_audio_file mismatch for {path}: got {result}'

print()
print(f'AUDIO_EXTENSIONS = {sorted(AUDIO_EXTENSIONS)}')

print()
print('=== _extract_audio_metadata() ===')
for fpath in [WAV_FILE, MP3_FILE, M4A_FILE]:
    meta = vp._extract_audio_metadata(fpath)
    p    = Path(fpath)
    ok   = meta.duration > 0 and not meta.is_silent
    print(f'  {"ok" if ok else "FAIL"}  {p.name:<25}  '
          f'duration={meta.duration:.1f}s  is_silent={meta.is_silent}  '
          f'title={meta.title}')
    assert meta.duration > 0,   f'{p.name}: duration=0 — ffprobe may be unavailable'
    assert not meta.is_silent,  f'{p.name}: is_silent=True — audio file should not be silent'

print()
print('[ASSERT OK] is_audio_file() and _extract_audio_metadata() correct for all files')


2026-04-24 15:03:21,952 - radiant_rag_mcp.ingestion.video_processor - INFO - Audio file metadata: 'lecture_en' (126s)
2026-04-24 15:03:22,035 - radiant_rag_mcp.ingestion.video_processor - INFO - Audio file metadata: 'lecture_en' (126s)


=== is_audio_file() ===
  ok  lecture_en.wav             is_audio_file=True  (expected True)
  ok  lecture_en.mp3             is_audio_file=True  (expected True)
  ok  lecture_en.m4a             is_audio_file=True  (expected True)
  ok  some_video.mp4             is_audio_file=False  (expected False)

AUDIO_EXTENSIONS = ['.aac', '.aif', '.aiff', '.flac', '.m4a', '.mp3', '.ogg', '.opus', '.wav', '.wma']

=== _extract_audio_metadata() ===
  ok  lecture_en.wav             duration=126.1s  is_silent=False  title=lecture_en
  ok  lecture_en.mp3             duration=126.1s  is_silent=False  title=lecture_en


2026-04-24 15:03:22,126 - radiant_rag_mcp.ingestion.video_processor - INFO - Audio file metadata: 'lecture_en' (126s)


  ok  lecture_en.m4a             duration=126.1s  is_silent=False  title=lecture_en

[ASSERT OK] is_audio_file() and _extract_audio_metadata() correct for all files


## 9  `ingest_audio` — WAV and MP3 ingestion  *(Whisper transcription)*

Ingests `lecture_en.wav` and `lecture_en.mp3` via the `ingest_audio` MCP tool.
Both files are routed through `process_local_audio()` → Whisper transcription
→ transcript chunks.  Asserts:
- `sources_processed == 2`
- `audio_sources == 2`
- `chunks_created > 0`


In [13]:
%%time
print('=== ingest_audio (lecture_en.wav + lecture_en.mp3 — Whisper path) ===')
r = run('ingest_audio', {
    'sources':      [WAV_FILE, MP3_FILE],
    'hierarchical': True,
    'summarize':    False,
})

assert_ok(r, 'sources_processed')

assert r.get('sources_processed') == 2, \
    f'Expected sources_processed=3, got {r.get("sources_processed")}'
assert r.get('audio_sources') == 2, \
    f'Expected audio_sources=2, got {r.get("audio_sources")}'
assert r.get('chunks_created', 0) > 0, \
    f'Expected chunks_created>0, got {r.get("chunks_created")}'

print(f'\nChunks created   : {r.get("chunks_created")}')
print(f'Documents stored : {r.get("documents_stored")}')
print(f'Audio sources    : {r.get("audio_sources")}')
print(f'Errors           : {r.get("errors")}')
print('[ASSERT OK] sources_processed==2, audio_sources==2, chunks_created>0')


2026-04-24 15:08:17,847 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: dc0f19c8931c458c9089356179157981


=== ingest_audio (lecture_en.wav + lecture_en.mp3 — Whisper path) ===
INFO:     127.0.0.1:34768 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:17,852 - mcp.client.streamable_http - INFO - Received session ID: dc0f19c8931c458c9089356179157981
2026-04-24 15:08:17,853 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:34770 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:34776 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:34784 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:17,871 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-24 15:08:17,981 - radiant_rag_mcp.ingestion.video_processor - INFO - Audio file metadata: 'lecture_en' (126s)
2026-04-24 15:08:17,982 - radiant_rag_mcp.ingestion.video_processor - INFO - Loading faster-whisper model 'base' on cuda (compute_type=int8)
2026-04-24 15:08:18,575 - radiant_rag_mcp.ingestion.video_processor - INFO - faster-whisper model ready.
2026-04-24 15:08:19,290 - faster_whisper - INFO - Processing audio with duration 02:06.062
2026-04-24 15:08:19,645 - faster_whisper - INFO - Detected language 'en' with probability 0.98
2026-04-24 15:08:22,445 - radiant_rag_mcp.ingestion.video_processor - INFO - Transcribed /tmp/radiant_audio_test/lecture_en.wav: 24 segments, language=en
2026-04-24 15:08:22,446 - radiant_rag_mcp.ingestion.video_processor - INFO - Produced 3 transcript chunks from 24 segments (source=/tmp/radiant_audio_test/lecture_en.wav)
2026-04-24 15:08:22,

INFO:     127.0.0.1:34792 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:25,717 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:08:25,732 - mcp.server.streamable_http - INFO - Terminating session: dc0f19c8931c458c9089356179157981


INFO:     127.0.0.1:34802 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "sources_processed": 2,
  "sources_failed": 0,
  "chunks_created": 6,
  "documents_stored": 16,
  "audio_sources": 2,
  "summaries": {},
  "errors": []
}
[OK]

Chunks created   : 6
Documents stored : 16
Audio sources    : 2
Errors           : []
[ASSERT OK] sources_processed==2, audio_sources==2, chunks_created>0
CPU times: user 6.36 s, sys: 344 ms, total: 6.7 s
Wall time: 7.95 s


In [14]:
%%time
# Also ingest the M4A to exercise a third audio format
print('=== ingest_audio (lecture_en.m4a — M4A format) ===')
r = run('ingest_audio', {
    'sources':      [M4A_FILE],
    'hierarchical': True,
    'summarize':    False,
})

assert_ok(r, 'sources_processed')
assert r.get('sources_processed') == 1, \
    f'Expected sources_processed=1, got {r.get("sources_processed")}'
assert r.get('audio_sources') == 1, \
    f'Expected audio_sources=1, got {r.get("audio_sources")}'

print(f'\nChunks created   : {r.get("chunks_created")}')
print(f'Audio sources    : {r.get("audio_sources")}')
print(f'Errors           : {r.get("errors")}')
print('[ASSERT OK] M4A ingested successfully')


2026-04-24 15:08:35,026 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: c1b10ab293ca493aa86104d8eb80ffa5


=== ingest_audio (lecture_en.m4a — M4A format) ===
INFO:     127.0.0.1:35970 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:35,031 - mcp.client.streamable_http - INFO - Received session ID: c1b10ab293ca493aa86104d8eb80ffa5
2026-04-24 15:08:35,033 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:35984 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:35988 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:35994 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:35,054 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-24 15:08:35,142 - radiant_rag_mcp.ingestion.video_processor - INFO - Audio file metadata: 'lecture_en' (126s)
2026-04-24 15:08:35,143 - radiant_rag_mcp.ingestion.video_processor - INFO - Loading faster-whisper model 'base' on cuda (compute_type=int8)
2026-04-24 15:08:35,733 - radiant_rag_mcp.ingestion.video_processor - INFO - faster-whisper model ready.
2026-04-24 15:08:36,369 - faster_whisper - INFO - Processing audio with duration 02:06.084
2026-04-24 15:08:36,558 - faster_whisper - INFO - Detected language 'en' with probability 0.98
2026-04-24 15:08:38,870 - radiant_rag_mcp.ingestion.video_processor - INFO - Transcribed /tmp/radiant_audio_test/lecture_en.m4a: 24 segments, language=en
2026-04-24 15:08:38,871 - radiant_rag_mcp.ingestion.video_processor - INFO - Produced 3 transcript chunks from 24 segments (source=/tmp/radiant_audio_test/lecture_en.m4a)
2026-04-24 15:08:38,

INFO:     127.0.0.1:55934 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:38,981 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:08:38,992 - mcp.server.streamable_http - INFO - Terminating session: c1b10ab293ca493aa86104d8eb80ffa5


INFO:     127.0.0.1:55950 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "sources_processed": 1,
  "sources_failed": 0,
  "chunks_created": 3,
  "documents_stored": 8,
  "audio_sources": 1,
  "summaries": {},
  "errors": []
}
[OK]

Chunks created   : 3
Audio sources    : 1
Errors           : []
[ASSERT OK] M4A ingested successfully
CPU times: user 3.66 s, sys: 248 ms, total: 3.91 s
Wall time: 4.03 s


In [15]:
# Verify rejection of a non-audio path
print('=== ingest_audio (invalid extension — should be rejected) ===')
r = run('ingest_audio', {
    'sources': ['/tmp/some_video.mp4'],
})

assert isinstance(r, dict), 'Expected dict response'
assert r.get('sources_failed') == 1, \
    f'Expected sources_failed=1 for unsupported extension, got: {r}'
assert any('not a recognised audio format' in e for e in r.get('errors', [])), \
    f'Expected format-rejection error message, got: {r.get("errors")}'

print('[ASSERT OK] Non-audio path correctly rejected before processing')


2026-04-24 15:08:47,426 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 75546d510a7746e8b7ca124952241d21


=== ingest_audio (invalid extension — should be rejected) ===
INFO:     127.0.0.1:51498 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:47,431 - mcp.client.streamable_http - INFO - Received session ID: 75546d510a7746e8b7ca124952241d21
2026-04-24 15:08:47,433 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:51514 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:51516 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:51518 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:47,453 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:51520 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:08:47,464 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:08:47,475 - mcp.server.streamable_http - INFO - Terminating session: 75546d510a7746e8b7ca124952241d21


INFO:     127.0.0.1:51536 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "sources_processed": 0,
  "sources_failed": 1,
  "chunks_created": 0,
  "documents_stored": 0,
  "audio_sources": 0,
  "summaries": {},
  "errors": [
    "/tmp/some_video.mp4: not a recognised audio format. Supported: .aac, .aif, .aiff, .flac, .m4a, .mp3, .ogg, .opus, .wav, .wma"
  ]
}
[ASSERT OK] Non-audio path correctly rejected before processing


## 10  Index stats after ingestion

Confirms the vector store and BM25 index grew after all three audio files
were ingested in section 9.


In [16]:
print('=== get_index_stats (after audio ingest) ===')
r = run('get_index_stats')
assert_ok(r)

if isinstance(r, dict):
    health = r.get('health', r)
    vi = health.get('vector_index', {})
    bi = health.get('bm25_index', {})
    print(f'\nVector doc count : {vi.get("document_count", "?")}')
    print(f'BM25   doc count : {bi.get("document_count", "?")}')
    assert (vi.get('document_count') or 0) > 0, \
        'Vector index is empty after ingestion'
    print('[ASSERT OK] Index contains documents')


2026-04-24 15:09:21,323 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 24184778d79140eeb85d62bed99cce8b


=== get_index_stats (after audio ingest) ===
INFO:     127.0.0.1:41074 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:21,329 - mcp.client.streamable_http - INFO - Received session ID: 24184778d79140eeb85d62bed99cce8b
2026-04-24 15:09:21,331 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:41078 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:41094 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:41106 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:21,350 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:41120 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:21,368 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:09:21,378 - mcp.server.streamable_http - INFO - Terminating session: 24184778d79140eeb85d62bed99cce8b


INFO:     127.0.0.1:41122 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:21,384 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 24,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 15,
      "unique_terms": 170,
      "avg_doc_length": 60.2,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 0,
    "average_latency_ms": null,
    "average_success_rate": 1.0,
    "history": []
  }
}
[OK]

Vector doc count : 24
BM25   doc count : 15
[ASSERT OK] Index contains documents


## 11  `search_knowledge` across audio content

Runs hybrid and BM25 searches against the ingested transcript chunks.
Results should reference content from the machine-learning lecture narration.


In [17]:
%%time
print('=== search_knowledge: hybrid ===')
r = run('search_knowledge', {
    'query': 'neural network layers neurons activation function',
    'mode':  'hybrid',
    'top_k': 6,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    print()
    print(f'  {"content_type":<24}  {"source":<25}  {"t/window":<10}  preview')
    print('  ' + '-' * 80)
    for hit in r['results'][:6]:
        meta    = hit.get('meta', hit.get('metadata', {}))
        ctype   = meta.get('content_type', '?')
        ts      = meta.get('start_time', '?')
        source  = Path(meta.get('source', '?')).name
        preview = hit.get('content', '')[:80].replace('\n', ' ')
        print(f'  {ctype:<24}  {source:<25}  {str(ts):<10}  {preview}...')


2026-04-24 15:09:32,269 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 066d4ff29e964e90a08c98183b86fbbb


=== search_knowledge: hybrid ===
INFO:     127.0.0.1:41042 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:32,274 - mcp.client.streamable_http - INFO - Received session ID: 066d4ff29e964e90a08c98183b86fbbb
2026-04-24 15:09:32,275 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:41052 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:41054 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:41068 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:32,294 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-24 15:09:32,297 - DenseRetrievalAgent - INFO - [0e36a994] [enabled=True] Initialized DenseRetrievalAgent
2026-04-24 15:09:32,413 - DenseRetrievalAgent - INFO - [50af74e2] [query_length=49 num_results=6 top_k=6] Dense retrieval completed
2026-04-24 15:09:32,414 - DenseRetrievalAgent - INFO - [50af74e2] [duration_ms=115.89 status=success] Execution completed
2026-04-24 15:09:32,415 - BM25RetrievalAgent - INFO - [e09bc380] [enabled=True] Initialized BM25RetrievalAgent
2026-04-24 15:09:32,425 - BM25RetrievalAgent - INFO - [6704c49f] [query_length=49 num_results=6 top_k=6] BM25 retrieval completed
2026-04-24 15:09:32,426 - BM25RetrievalAgent - INFO - [6704c49f] [duration_ms=10.06 status=success] Execution completed
2026-04-24 15:09:32,427 - RRFAgent - INFO - [1f64ed7e] [enabled=True] Initialized RRFAgent
2026-04-24 15:09:32,428 - RRFAgent - INFO - [9c0440f4] [num_runs=2 total_doc

INFO:     127.0.0.1:41076 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:32,442 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:09:32,453 - mcp.server.streamable_http - INFO - Terminating session: 066d4ff29e964e90a08c98183b86fbbb


INFO:     127.0.0.1:41086 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:32,458 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "neural network layers neurons activation function",
  "mode": "hybrid",
  "results": [
    {
      "doc_id": "f86d1205ead2de835b6d904f79b928d0e061db5943124243e6e0e27cec77d9da",
      "content": "compressed data into fewer dimensions while preserving structure. Reinforcement learning trains an agent to maximize cumulative reward through interaction with an environment. Neural networks are a core architecture behind modern machine learning. A neural network consists of layers of interconnected neurons. Each neuron applies a weighted sum followed by a nonlinear activation function. Deep learning refers to neural networks with many hidden layers. Convolutional networks excel at image recognition",
      "meta": {
        "is_youtube": false,
        "title": "lecture_en",
        "file_type": "video",
        "chunk_index": 1,
        "source": "/tmp/radiant_audio_test/lecture_en.mp3",
        "child_total": 2,
        "content_type": "transcript",
        "total_chunks": 3,


In [18]:
%%time
print('=== search_knowledge: bm25 ===')
r = run('search_knowledge', {
    'query': 'supervised learning labelled training data classification',
    'mode':  'bm25',
    'top_k': 5,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    print()
    for hit in r['results'][:4]:
        meta  = hit.get('meta', hit.get('metadata', {}))
        ctype = meta.get('content_type', '?')
        score = hit.get('score', 0)
        src   = Path(meta.get('source', '?')).name
        print(f'  [{ctype}]  {src}  score={score:.4f}')
        print(f'    {hit.get("content", "")[:100]}')
        print()


2026-04-24 15:09:45,489 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 9e254a52a7ec4370b2720df40e5be4de


=== search_knowledge: bm25 ===
INFO:     127.0.0.1:45100 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:45,494 - mcp.client.streamable_http - INFO - Received session ID: 9e254a52a7ec4370b2720df40e5be4de
2026-04-24 15:09:45,496 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:45102 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:45114 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:45130 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:45,514 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-24 15:09:45,517 - BM25RetrievalAgent - INFO - [00594458] [enabled=True] Initialized BM25RetrievalAgent
2026-04-24 15:09:45,529 - BM25RetrievalAgent - INFO - [f04d32ed] [query_length=57 num_results=5 top_k=5] BM25 retrieval completed
2026-04-24 15:09:45,530 - BM25RetrievalAgent - INFO - [f04d32ed] [duration_ms=7.35 status=success] Execution completed


INFO:     127.0.0.1:45144 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:45,543 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:09:45,553 - mcp.server.streamable_http - INFO - Terminating session: 9e254a52a7ec4370b2720df40e5be4de


INFO:     127.0.0.1:45152 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-24 15:09:45,557 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "supervised learning labelled training data classification",
  "mode": "bm25",
  "results": [
    {
      "doc_id": "084408769a9043120e84cb6de96d5df4e1d7e2d4b15c39d25a2659b1cdd9d902",
      "content": "introduction to machine learning. Machine learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed. There are three main paradigms. Supervised learning, unsupervised learning, and reinforcement learning. In supervised learning, models are trained on labeled examples. The model learns to map inputs to outputs by minimizing a loss function. Common supervised tasks include classification and regression. Unsupervised learning discovers hidden",
      "meta": {
        "end_time": 60.0,
        "child_total": 2,
        "is_youtube": false,
        "chunk_index": 0,
        "total_chunks": 3,
        "content_type": "transcript",
        "video_id": "",
        "doc_level": "child",
        "language": "en",
   

## 12  `query_knowledge` grounded in audio content  `[LLM]`

> **Requires a configured LLM backend and a valid `OLLAMA_API_KEY`.**

Sends a natural-language question to the full RAG pipeline.
The retrieval context is drawn from the audio transcript chunks.


In [19]:
%%time
if not skip_llm():
    print('=== query_knowledge (audio-grounded) ===')
    r = run('query_knowledge', {
        'question': (
            'What are the three main paradigms of machine learning described '
            'in the lecture, and how does each one work?'
        ),
        'mode': 'hybrid',
    })
    assert_ok(r, 'answer')
    print()
    print('Answer:')
    print('-' * 60)
    print(r['answer'][:900])
    print('-' * 60)
    print(f'Warnings : {r.get("warnings", [])}')


2026-04-24 15:10:16,219 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: d4e8d058b7704e93b40e1bac3527d56d


=== query_knowledge (audio-grounded) ===
INFO:     127.0.0.1:55724 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:10:16,225 - mcp.client.streamable_http - INFO - Received session ID: d4e8d058b7704e93b40e1bac3527d56d
2026-04-24 15:10:16,226 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:55728 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:55730 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55734 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:10:16,243 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-24 15:10:16,246 - radiant_rag_mcp.orchestrator - INFO - [f89bb36b-4cad-422e-814a-ef58b98fb873] Starting agentic pipeline for query: What are the three main paradigms of machine learning described in the lecture, and how does each on...
2026-04-24 15:10:16,248 - radiant_rag_mcp.orchestrator - INFO - [f89bb36b-4cad-422e-814a-ef58b98fb873] Retrieval mode: hybrid, fast_path: False
2026-04-24 15:10:16,299 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Initialized OpenAI-compatible LLM backend: model=gemma3:12b-cloud, base_url=https://ollama.com/v1
2026-04-24 15:10:17,838 - QueryRewriteAgent - INFO - [f89bb36b-4cad-422e-814a-ef58b98fb873] [original=What are the three main paradigms of machine learn rewritten=Identify three primary machine learning paradigms ] Query rewritten
2026-04-24 15:10:17,839 - QueryRewriteAgent - INFO - [f89bb36b-4cad-422e-814a-ef58b98fb873] [duration

INFO:     127.0.0.1:55744 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:10:20,530 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:10:20,540 - mcp.server.streamable_http - INFO - Terminating session: d4e8d058b7704e93b40e1bac3527d56d


INFO:     127.0.0.1:55750 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-24 15:10:20,545 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "run_id": "f89bb36b-4cad-422e-814a-ef58b98fb873",
  "answer": "The three main paradigms of machine learning are supervised learning, unsupervised learning, and reinforcement learning [DOC 1, DOC 2, DOC 3].\n\n*   **Supervised learning** trains models on labeled examples to map inputs to outputs by minimizing a loss function. Common tasks include classification and regression [DOC 1, DOC 2].\n*   **Unsupervised learning** discovers hidden patterns in unlabeled data. This includes clustering algorithms and dimensionality reduction techniques like principle component analysis [DOC 1, DOC 2].\n*   **Reinforcement learning** trains an agent to maximize cumulative reward through interaction with an environment [DOC 1, DOC 2].",
  "success": true,
  "error": null,
  "original_query": "What are the three main paradigms of machine learning described in the lecture, and how does each one work?",
  "decomposed_queries": [],
  "num_retrieved_docs": 7,
  "warnings": [],
  "metrics": {
    "run_

## 13  `VideoSummarizationAgent` — `brief` summary  `[LLM]`

> **Requires a configured LLM backend.**

Runs `VideoSummarizationAgent` directly on the WAV transcript chunks.
`summary_detail='brief'` produces:
- Chapter summary: 1 short paragraph
- Overall summary: 1 short paragraph


In [20]:
%%time
if not skip_llm():
    from radiant_rag_mcp.app import create_app
    from radiant_rag_mcp.agents.video_summarization import VideoSummarizationAgent
    from radiant_rag_mcp.config import VideoSummarizationConfig

    # Build app and extract the shared LLM client once; reused in sections 14-16
    _app = create_app(CONFIG_PATH)
    _llm = _app._orchestrator._llm

    # Retrieve transcript chunks for the WAV file
    hits   = _app.search('lecture_en', mode='dense', top_k=50, show_results=False)
    chunks = [
        doc for doc, _ in hits
        if Path(doc.meta.get('source', '')).name == 'lecture_en.wav'
    ]
    print(f'Chunks retrieved for lecture_en.wav: {len(chunks)}')

    if not chunks:
        print('[NOTE] No chunks found for lecture_en.wav — run section 9 first.')
    else:
        cfg        = VideoSummarizationConfig(summary_detail='brief')
        agent      = VideoSummarizationAgent(llm=_llm, config=cfg)
        _res_brief = agent.summarize_video(WAV_FILE, chunks)

        print(f'Title      : {_res_brief.title}')
        print(f'is_silent  : {_res_brief.is_silent}')
        print(f'Duration   : {_res_brief.duration_seconds:.0f}s')
        print(f'Chapters   : {len(_res_brief.chapters)}')
        print(f'Key topics : {_res_brief.key_topics}')
        print()
        print('Overall summary (brief):')
        print('-' * 60)
        print(_res_brief.summary)
        print('-' * 60)
        print(f'Word count (overall) : {len(_res_brief.summary.split())}')


2026-04-24 15:11:23,569 - radiant_rag_mcp.app - INFO - Initializing Radiant RAG...
2026-04-24 15:11:23,570 - radiant_rag_mcp.llm.client - INFO - Using new backend configuration system
2026-04-24 15:11:23,571 - radiant_rag_mcp.llm.backends.factory - INFO - Creating LLM backend: type=ollama
2026-04-24 15:11:23,571 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Configured OpenAI-compatible LLM backend: model=gemma3:12b-cloud, base_url=https://ollama.com/v1
2026-04-24 15:11:23,572 - radiant_rag_mcp.llm.backends.factory - INFO - Creating embedding backend: type=local
2026-04-24 15:11:23,573 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Initializing sentence-transformers embedding backend: sentence-transformers/all-MiniLM-L12-v2
[transformers] The following layers were not sharded: encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.self.query.weight, embeddings.LayerNorm.bias, pooler.dense.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.l

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/llm/backends/embedding_backends.py:104: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._embedding_dim = self._model.get_sentence_embedding_dimension()
2026-04-24 15:11:26,667 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Loaded sentence-transformers embedding model: sentence-transformers/all-MiniLM-L12-v2 (dim=384, device=cuda)
2026-04-24 15:11:26,668 - radiant_rag_mcp.llm.backends.factory - INFO - Creating reranking backend: type=local
2026-04-24 15:11:26,669 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Initializing cross-encoder reranking backend: cross-encoder/ms-marco-MiniLM-L12-v2
[transformers] The following layers were not sharded: bert.encoder.layer.*.attention.output.dense.weight, bert.encoder.layer.*.attention.self.query.weight, bert.encoder.layer.*.output.LayerNorm.bias, bert.pooler.dense.weight, bert.pooler.dense.bias, bert.e

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-04-24 15:11:28,919 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Loaded cross-encoder reranking model: cross-encoder/ms-marco-MiniLM-L12-v2 (device=cuda)
2026-04-24 15:11:28,920 - radiant_rag_mcp.storage.factory - INFO - Creating Chroma vector store
2026-04-24 15:11:28,928 - radiant_rag_mcp.storage.chroma_store - INFO - Initialized ChromaDB store at 'data/chroma_db' with collection 'radiant_docs'
2026-04-24 15:11:28,928 - radiant_rag_mcp.utils.conversation - INFO - ConversationStore: storage backend is 'chroma' — using in-memory storage
2026-04-24 15:11:28,930 - PlanningAgent - INFO - [9dea4f04] [enabled=True] Initialized PlanningAgent
2026-04-24 15:11:28,931 - QueryDecompositionAgent - INFO - [ba0edae1] [enabled=True] Initialized QueryDecompositionAgent
2026-04-24 15:11:28,933 - QueryRewriteAgent - INFO - [9203f1a6] [enabled=True] Initialized QueryRewriteAgent
2026-04-24 15:11:28,934 - QueryExpansionAgent - INFO - [7f1187da] [enabled=True] Initialized QueryExpansionA

Chunks retrieved for lecture_en.wav: 5


2026-04-24 15:11:36,166 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: summarized 'lecture_en' → 1 chapters, 7 topics, model=unknown


Title      : lecture_en
is_silent  : False
Duration   : 126s
Chapters   : 1
Key topics : ['Machine learning basics', 'Supervised learning', 'Unsupervised learning', 'Reinforcement learning', 'Neural networks', 'Deep learning architectures', 'Training & evaluation']

Overall summary (brief):
------------------------------------------------------------
This video provides a concise overview of machine learning, its core concepts, and foundational architectures. The 126-second lecture aims to introduce viewers to the field, laying out the essential building blocks for further exploration.

The video begins by defining machine learning as a subset of AI focused on learning from data. It then outlines the three primary learning paradigms: supervised, unsupervised, and reinforcement learning, detailing how each utilizes data and common techniques. The explanation then transitions to neural networks, describing their structure with interconnected neurons, weighted sums, and activation functio

## 14  `VideoSummarizationAgent` — `standard` summary  `[LLM]`

> **Requires a configured LLM backend.**

`summary_detail='standard'`:
- Chapter summary: 1–2 structured paragraphs
- Overall summary: 2–3 paragraphs


In [21]:
%%time
if not skip_llm():
    cfg          = VideoSummarizationConfig(summary_detail='standard')
    agent        = VideoSummarizationAgent(llm=_llm, config=cfg)
    _res_standard = agent.summarize_video(WAV_FILE, chunks)

    print('Overall summary (standard):')
    print('-' * 60)
    print(_res_standard.summary)
    print('-' * 60)
    print(f'Word count (overall) : {len(_res_standard.summary.split())}')
    print()
    print('Chapter breakdowns:')
    for ch in _res_standard.chapters:
        print(f'  Ch {ch.index + 1}: [{ch.start:.0f}s – {ch.end:.0f}s]  {ch.title}')
        for ln in ch.summary.split('\n')[:4]:
            print(f'    {ln[:110]}')
        print()


2026-04-24 15:12:51,584 - VideoSummarizationAgent - INFO - [f8d28f69] [enabled=True] Initialized VideoSummarizationAgent
2026-04-24 15:12:51,585 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: 5 chunks → 1 chapters (source=/tmp/radiant_audio_test/lecture_en.wav, type=transcript)
2026-04-24 15:12:58,418 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: summarized 'lecture_en' → 1 chapters, 8 topics, model=unknown


Overall summary (standard):
------------------------------------------------------------
This video serves as a concise introduction to the field of machine learning. Covering a broad range of foundational concepts, the 126-second lecture aims to provide a high-level overview of the core principles and techniques underpinning this rapidly evolving area of artificial intelligence.

The video systematically presents the central ideas of machine learning, beginning with its definition as a data-driven approach to computing. It then explores the three main learning paradigms: supervised, unsupervised, and reinforcement learning, outlining the distinct goals and methodologies within each. The discussion then transitions to neural networks, detailing their basic structure—layers of interconnected neurons performing weighted sums and utilizing activation functions—and introducing the concept of deep learning through multilayer networks. Specific architectures like convolutional networks, recu

## 15  `VideoSummarizationAgent` — `detailed` summary  `[LLM]`

> **Requires a configured LLM backend.**

`summary_detail='detailed'`:
- Chapter summary: 2–3 paragraphs (8–15 sentences)
- Overall summary: 3–4 structured paragraphs


In [22]:
%%time
if not skip_llm():
    cfg          = VideoSummarizationConfig(summary_detail='detailed')
    agent        = VideoSummarizationAgent(llm=_llm, config=cfg)
    _res_detailed = agent.summarize_video(WAV_FILE, chunks)

    print('Overall summary (detailed):')
    print('-' * 60)
    print(_res_detailed.summary)
    print('-' * 60)
    print(f'Word count (overall) : {len(_res_detailed.summary.split())}')
    print()
    print(f'Chapters : {len(_res_detailed.chapters)}')
    for ch in _res_detailed.chapters:
        print(f'  Ch {ch.index + 1}: [{ch.start:.0f}s – {ch.end:.0f}s]  {ch.title}')


2026-04-24 15:13:11,874 - VideoSummarizationAgent - INFO - [8ad224a1] [enabled=True] Initialized VideoSummarizationAgent
2026-04-24 15:13:11,876 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: 5 chunks → 1 chapters (source=/tmp/radiant_audio_test/lecture_en.wav, type=transcript)
2026-04-24 15:13:19,664 - radiant_rag_mcp.agents.video_summarization - INFO - VideoSummarizationAgent: summarized 'lecture_en' → 1 chapters, 8 topics, model=unknown


Overall summary (detailed):
------------------------------------------------------------
This video presents a concise introductory lecture on machine learning, covering its core concepts and architectural foundations. The video, lasting 126 seconds, aims to provide a broad overview suitable for viewers new to the field. Its primary subject is defining machine learning and exploring its major paradigms and key technologies.

The lecture progresses chronologically through the foundational elements of machine learning. It begins by defining machine learning as a branch of artificial intelligence that allows computers to learn from data without explicit programming. The three main learning paradigms – supervised, unsupervised, and reinforcement learning – are then explained, with specific examples like classification, regression, clustering, and dimensionality reduction (including PCA). The video then shifts focus to the architectural aspects, introducing neural networks, deep learning, a

## 16  Summary detail levels — word-count comparison  `[LLM]`

> **Requires a configured LLM backend.**

Runs all three detail levels and prints a side-by-side table covering:
- **Chapter** average word count
- **Overall** total word count
- **Chapter count**


In [23]:
%%time
if not skip_llm():
    detail_results = {}
    for detail, cached in [
        ('brief',    globals().get('_res_brief')),
        ('standard', globals().get('_res_standard')),
        ('detailed', globals().get('_res_detailed')),
    ]:
        if cached is not None:
            detail_results[detail] = cached
            print(f'  {detail:<10}  (using cached result from sections 13-15)')
        else:
            cfg   = VideoSummarizationConfig(summary_detail=detail)
            agent = VideoSummarizationAgent(llm=_llm, config=cfg)
            detail_results[detail] = agent.summarize_video(WAV_FILE, chunks)
            print(f'  {detail:<10}  (freshly generated)')

    print()
    HDR = f'  {"Detail":<10}  {"Chapter avg":>12}  {"Overall total":>14}  {"Chapters":>8}'
    print(HDR)
    print('  ' + '-' * (len(HDR) - 2))

    for detail, res in detail_results.items():
        overall_wc = len(res.summary.split())
        ch_wcs     = [len(ch.summary.split()) for ch in res.chapters]
        ch_avg     = (sum(ch_wcs) // max(1, len(ch_wcs))) if ch_wcs else 0
        print(f'  {detail:<10}  {ch_avg:>12,}  {overall_wc:>14,}  {len(res.chapters):>8}')


  brief       (using cached result from sections 13-15)
  standard    (using cached result from sections 13-15)
  detailed    (using cached result from sections 13-15)

  Detail       Chapter avg   Overall total  Chapters
  --------------------------------------------------
  brief                260             124         1
  standard             220             189         1
  detailed             223             173         1
CPU times: user 487 µs, sys: 0 ns, total: 487 µs
Wall time: 480 µs


## 17  Cleanup

- Clears the ChromaDB vector index
- Deletes the temporary audio files from `/tmp/radiant_audio_test/`

Set `RUN_CLEAR = False` to keep the index for further experimentation.


In [24]:
RUN_CLEAR = True   # Set to False to keep the index intact

if RUN_CLEAR:
    print('=== clear_index ===')
    r = run('clear_index', {'confirm': True})
    if isinstance(r, dict):
        if r.get('cleared'):
            print('[OK] Index cleared.')
        elif ('_ensure_index' in r.get('message', '')
              or 'clear failed' in r.get('message', '').lower()):
            print('[OK] Collection dropped (known ChromaDB reinit pattern).')
        else:
            raise AssertionError(f'Unexpected clear failure: {r}')

    stats = asyncio.run(_call('get_index_stats'))
    if isinstance(stats, dict):
        vi = stats.get('health', stats).get('vector_index', {})
        print(f'Vector doc count post-clear: {vi.get("document_count", "?")}')
else:
    print('RUN_CLEAR=False — index preserved.')


2026-04-24 15:13:52,345 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 6076c99dc9514b268e6adcc6567e0164


=== clear_index ===
INFO:     127.0.0.1:44126 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:13:52,352 - mcp.client.streamable_http - INFO - Received session ID: 6076c99dc9514b268e6adcc6567e0164
2026-04-24 15:13:52,353 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:44142 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:44146 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:44150 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:13:52,375 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-24 15:13:52,404 - radiant_rag_mcp.storage.chroma_store - INFO - Dropped Chroma collection 'radiant_docs'
2026-04-24 15:13:52,406 - radiant_rag_mcp.app - ERROR - Failed to clear index: 'ChromaVectorStore' object has no attribute '_ensure_index'


...

INFO:     127.0.0.1:44210 - "POST /mcp HTTP/1.1" 200 OK


2026-04-24 15:13:52,563 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-24 15:13:52,574 - mcp.server.streamable_http - INFO - Terminating session: 17ac87687cfc440eb66541b757e611e7


INFO:     127.0.0.1:44220 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-24 15:13:52,579 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


Vector doc count post-clear: 0


In [25]:
import shutil

_test_dir = Path('/tmp/radiant_audio_test')
if _test_dir.exists():
    shutil.rmtree(_test_dir)
    print(f'Removed test audio directory: {_test_dir}')
else:
    print(f'Directory not found (already cleaned?): {_test_dir}')

print('Cleanup complete')


Removed test audio directory: /tmp/radiant_audio_test
Cleanup complete


## Summary

Run this cell at any point for a status overview.


In [26]:
print('=' * 65)
print('  Radiant RAG MCP — Audio Ingestion & Summarization Test')
print('  Run complete')
print('=' * 65)
print()
print('  Synthetic test audio  (espeak TTS, ~30 s narration)')
print('    lecture_en.wav  — WAV 16 kHz mono')
print('    lecture_en.mp3  — MP3 128 kbps')
print('    lecture_en.m4a  — AAC / M4A 128 kbps')
print()
print('  Audio metadata')
print('    ok  is_audio_file()           .wav/.mp3/.m4a  == True')
print('    ok  is_audio_file()           .mp4            == False')
print('    ok  _extract_audio_metadata() duration > 0, is_silent=False')
print()
print('  Audio ingestion  (ingest_audio MCP tool)')
print('    ok  lecture_en.wav + .mp3  -> Whisper -> transcript chunks')
print('    ok  lecture_en.m4a         -> Whisper -> transcript chunks')
print('    ok  .mp4 extension         -> rejected before processing')
print()
print('  Search')
print('    ok  search_knowledge hybrid  (content_type=transcript)')
print('    ok  search_knowledge bm25')
print()
print('  LLM cells  [L = requires OLLAMA_API_KEY]')
print('   [L]  query_knowledge grounded in audio transcripts')
print('   [L]  VideoSummarizationAgent  brief')
print('   [L]  VideoSummarizationAgent  standard')
print('   [L]  VideoSummarizationAgent  detailed')
print('   [L]  Word-count table  (chapter avg / overall total)')
print()
print(f'  LLM key available : {HAS_LLM_KEY}')


  Radiant RAG MCP — Audio Ingestion & Summarization Test
  Run complete

  Synthetic test audio  (espeak TTS, ~30 s narration)
    lecture_en.wav  — WAV 16 kHz mono
    lecture_en.mp3  — MP3 128 kbps
    lecture_en.m4a  — AAC / M4A 128 kbps

  Audio metadata
    ok  is_audio_file()           .wav/.mp3/.m4a  == True
    ok  is_audio_file()           .mp4            == False
    ok  _extract_audio_metadata() duration > 0, is_silent=False

  Audio ingestion  (ingest_audio MCP tool)
    ok  lecture_en.wav + .mp3  -> Whisper -> transcript chunks
    ok  lecture_en.m4a         -> Whisper -> transcript chunks
    ok  .mp4 extension         -> rejected before processing

  Search
    ok  search_knowledge hybrid  (content_type=transcript)
    ok  search_knowledge bm25

  LLM cells  [L = requires OLLAMA_API_KEY]
   [L]  query_knowledge grounded in audio transcripts
   [L]  VideoSummarizationAgent  brief
   [L]  VideoSummarizationAgent  standard
   [L]  VideoSummarizationAgent  detailed
   [L]  W